# LLM-assisted classification of the abstracts (prompt version 2.1)

Each abstract retained by the screening (`data/articles/screened_articles.csv`, n = 1,600) is coded on the two dimensions of the strategic AI framework of Huang and Rust (2021): the **marketing stage** that the AI supports (research, strategy or action, with the 4Ps/4Cs area for actions) and the **AI intelligence** by its purpose (mechanical, thinking or feeling). Together they give the nine strategic elements of Table 1 of the article. The prompt (`mdm/prompt.py`) holds the definitions, the decision rules R1-R7, the reference applications of Tables 2 and 3 of the article and 13 worked examples.

**How the calls are made** (`mdm/classify.py`, `mdm/llm_client.py`)
- The API and the model are set only in the configuration cell below; every row of the output keeps the provider, the model and the exact model version returned by the API. Batches saved by a different provider or model are not mixed in: the run stops and asks for them to be moved.
- Several abstracts per request. The model returns `{"results": [...]}`, one entry per study id, constrained by a JSON schema with the allowed labels. Studies left out of an answer are sent again in smaller batches.
- Gemini: temperature 0 and a fixed pause between requests; on a rate limit the client waits the time the API asks for; if the free-tier daily quota ends, the run stops and continues from the next batch when the cell is run again. OpenAI (reasoning models): only the default temperature is accepted, so it is omitted, and the reasoning effort is set in the configuration cell.
- Every batch is saved as soon as it is done.

**API key**: put `LLM_API_KEY=...` in a file named `.env` in the repository folder (it is in `.gitignore`). Never write the key in the notebook.

**Order**: (1) pilot with 50 records, (2) authors code the same 50 records blind and compute the agreement, (3) full run.

In [30]:
import pandas as pd
from mdm import classify, prompt
from mdm.llm_client import load_api_key

# ---- LLM used for the classification -------------------------------------------------------------
# OpenAI API (Chat Completions). To use another model, change MODEL; to use another API, change PROVIDER
# (a new API also needs its request method in mdm/llm_client.py). Nothing else in the project names the model.
PROVIDER = 'openai'
MODEL = 'gpt-5.4-nano'
TEMPERATURE = None        # reasoning models accept only the default temperature (use 0.0 with Gemini)
REASONING_EFFORT = 'low'  # OpenAI reasoning models: 'minimal', 'low', 'medium' or 'high'
MIN_INTERVAL = 2          # seconds between requests
PILOT_BATCH = 10          # abstracts per request in the pilot
FULL_BATCH = 25           # abstracts per request in the full run

load_api_key()            # stops here if LLM_API_KEY is missing from .env
print(f'{PROVIDER} / {MODEL}: API key found')

openai / gpt-5.4-nano: API key found


### The prompt

In [31]:
system, phash = classify.system_prompt()
print(f'Prompt version {prompt.PROMPT_VERSION}, hash {phash}, {len(system):,} characters (about {len(system) // 4:,} tokens)')
print(f'Saved in data/framework/classification_prompt_v{prompt.PROMPT_VERSION}.txt')

Prompt version 2.1, hash 0e2ea495ba43, 42,949 characters (about 10,737 tokens)
Saved in data/framework/classification_prompt_v2.1.txt


### 1. Pilot (50 records, seed 2026)

In [32]:
pilot = classify.run('pilot', PROVIDER, MODEL, PILOT_BATCH, temperature=TEMPERATURE, min_interval=MIN_INTERVAL,
                 reasoning_effort=REASONING_EFFORT)

pilot: 50 records, 5 batches of up to 10; openai / gpt-5.4-nano; prompt v2.1 (0e2ea495ba43)
  batch 1/5: 10 records, 1 call(s), 13,412 tokens, 10s elapsed
  batch 2/5: 10 records, 1 call(s), 13,149 tokens, 21s elapsed
  batch 3/5: 10 records, 1 call(s), 13,835 tokens, 35s elapsed
  batch 4/5: 10 records, 1 call(s), 13,097 tokens, 45s elapsed
  batch 5/5: 10 records, 1 call(s), 13,489 tokens, 54s elapsed

pilot: 50 records classified
Answers with problems: 0
Evidence not found verbatim in the abstract: 5
ai_intelligence     Feeling AI  Mechanical AI  Not identified  Thinking AI  All
stage                                                                          
Marketing action             3              1               3            7   14
Marketing research           5              0               4            5   14
Marketing strategy           0              1               0            7    8
Not identified               0              0              13            1   14
All        

In [33]:
pilot = pd.read_csv('data/articles/llm_classification/pilot/classification_pilot.csv', dtype=str, keep_default_na=False)
print('Model version(s):', pilot['model_version'].unique())
print('Answers with warnings:', (pilot['parse_errors'] != '').sum())
print('Evidence not found verbatim in the abstract:', (pilot['evidence_in_abstract'] == 'False').sum())
display(pd.crosstab(pilot['stage'], pilot['ai_intelligence'], margins=True))
pd.set_option('display.max_colwidth', 160)
pilot[['Title', 'strategic_element', 'marketing_mix', 'evidence', 'rationale', 'confidence']].head(10)

Model version(s): ['gpt-5.4-nano-2026-03-17']
Answers with warnings: 0
Evidence not found verbatim in the abstract: 5


ai_intelligence,Feeling AI,Mechanical AI,Not identified,Thinking AI,All
stage,,,,,
Marketing action,3,1,3,7,14
Marketing research,5,0,4,5,14
Marketing strategy,0,1,0,7,8
Not identified,0,0,13,1,14
All,8,2,20,20,50


,Title,strategic_element,marketing_mix,evidence,rationale,confidence
0,How Generative AI Shapes Sustainable Wine Consumption and The Role of Anthropomorphism and Trust in the Lebanese Market,Relationalization,Promotion/Communication,Study 2 (N = 198) utilized a simulated AI–consumer chatbot interaction in a wine purchase scenario.,"The study tests an AI communicator (simulated AI–consumer chatbot) that influences consumers’ responses and sustainable wine choices, which is an AI deploym...",medium
1,Something approaching science? Cluster analysis procedures in the CRM era,Market analysis,,This paper reports the findings of a study which subjected a data set designed for segmentation purposes to a series of rigorous validity and reliability tests,"Cluster analysis procedures are evaluated for their reliability and validity when used for segmentation analytics in CRM, i.e., marketing research/analysis ...",medium
2,Monetizing the iot revolution,Personalization,Price/Cost,"Data convergence creates context-based-intelligence, which enables a shift from using consumer profiles for targeted advertising to individualized offers an...","The study describes IoT-enabled context-based intelligence used to provide individualized offers and pricing, which supports marketing action related to pri...",low
3,Evaluation of Green Marketing Strategies by Considering Sustainability Criteria,Targeting,,"Then, green marketing strategies in the Iranian food industry are ranked using several fuzzy multiple criteria decision-making methods",The study ranks and prioritizes green marketing strategies using multi-criteria decision-making and fuzzy methods; this is marketing strategy planning (sele...,medium
4,"Value co-creation in industrial AI: The interactive role of B2B supplier, customer and technology provider",Not identified,,"This research explores the interactive role of supplier, customer and technology company in business-to-business (B2B) marketing when they develop and use i...","The study uses interviews to identify value types and capabilities in B2B value co-creation with industrial AI, focusing on roles and collaborative value ra...",low
5,"AI-powered marketing: What, where, and how?",Not identified,,This research investigates how AI is currently applied across different marketing functions and its potential future evolution and impact on marketing proce...,This is a conceptual/research-agenda article describing AI applications across marketing functions rather than reporting an empirical AI use for a specific ...,low
6,"Progress toward the ""factory of the future""",Not identified,,"we wanted to identify the kinds of progress being made by these manufacturers and the barriers that they, and those who are contemplating such process innov...","The study concerns organizational adoption of factory automation technologies (manufacturing context) and organizational barriers/measurement capabilities, ...",low
7,When humor beats believability: How AI-generated imagery shapes pet-inclusive restaurant choice,Relationalization,Promotion/Communication,"Overall, the findings indicate two context-specific appraisal patterns: traditional pet-only imagery supports credibility-oriented responses, whereas AI-gen...",The study experimentally tests AI-generated imagery used in pet-inclusive restaurant marketing to shape consumers’ affective appraisal (humor) and intention...,high
8,Automated psychographic consumer segmentation: a text classification approach,Segmentation,,This paper demonstrates that automated segmentation of consumers based on Self-Determination Theory (SDT) is feasible using relatively simple text classific...,"The purpose is automated psychographic consumer segmentation from text interviews, which supports marketing strategy segmentation. Pattern discovery to form...",high
9,From Perception to Purchase: How AI Literacy Shapes Consumer Decisions in AI-Generated Sponsored Vlogs Across Products and Services,Relationalization,Promotion/Commu

### 2. Human check of the pilot

`data/articles/llm_classification/pilot/pilot_human_coding.csv` has the same 50 records without the model's answers. Each author fills `stage_human`, `ai_intelligence_human` and, for marketing actions, `marketing_mix_human` with the labels of the prompt. The cell below then reports the percent agreement and Cohen's kappa (report them in section 3.3).

In [ ]:
classify.agreement('data/articles/llm_classification/pilot/classification_pilot.csv',
                   'data/articles/llm_classification/pilot/pilot_human_coding.csv')

### 3. Full run (all screened records)

In [34]:
full = classify.run('full', PROVIDER, MODEL, FULL_BATCH, temperature=TEMPERATURE, min_interval=MIN_INTERVAL,
                 reasoning_effort=REASONING_EFFORT)

full: 1600 records, 64 batches of up to 25; openai / gpt-5.4-nano; prompt v2.1 (0e2ea495ba43)
  batch 1/64: 25 records, 1 call(s), 23,109 tokens, 36s elapsed
  batch 2/64: 25 records, 1 call(s), 19,755 tokens, 57s elapsed
  batch 3/64: 25 records, 1 call(s), 20,957 tokens, 79s elapsed
  batch 4/64: 25 records, 1 call(s), 21,598 tokens, 101s elapsed
  batch 5/64: 25 records, 1 call(s), 22,192 tokens, 132s elapsed
  batch 6/64: 25 records, 1 call(s), 24,691 tokens, 164s elapsed
  batch 7/64: 25 records, 1 call(s), 20,351 tokens, 184s elapsed
  batch 8/64: 25 records, 1 call(s), 20,906 tokens, 205s elapsed
  batch 9/64: 25 records, 1 call(s), 22,763 tokens, 237s elapsed
  batch 10/64: 25 records, 1 call(s), 20,851 tokens, 258s elapsed
  batch 11/64: 25 records, 1 call(s), 20,490 tokens, 281s elapsed
  batch 12/64: 25 records, 1 call(s), 22,562 tokens, 315s elapsed
  batch 13/64: 25 records, 1 call(s), 22,076 tokens, 349s elapsed
  batch 14/64: 25 records, 1 call(s), 22,578 tokens, 382s el

In [35]:
classify.status()

pilot: 5 batches saved, 50 records
full: 64 batches saved, 1600 records
